In [ ]:
# ============================================
# Context-Aware Chatbot Using LangChain and RAG
# Complete Implementation for Jupyter Notebook
# ============================================

# Cell 1: Install required packages
!pip install langchain langchain-community langchain-groq chromadb streamlit pypdf faiss-cpu sentence-transformers tiktoken python-dotenv -q

# Cell 2: Import all necessary libraries
import os
import warnings
from typing import List, Dict, Any
import streamlit as st

# LangChain imports
from langchain.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.memory import ConversationBufferMemory, VectorStoreRetrieverMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import Document
from langchain_community.chat_models import ChatOllama
from langchain_groq import ChatGroq
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Utilities
from dotenv import load_dotenv
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

# Cell 3: Setup and configuration
def setup_environment():
    """Setup environment variables and configurations"""
    
    # Create .env file template
    env_template = """
# API Keys
GROQ_API_KEY=your_groq_api_key_here
OPENAI_API_KEY=your_openai_api_key_here

# LangChain settings
LANGCHAIN_TRACING_V2=false
LANGCHAIN_PROJECT=context-aware-chatbot
"""
    
    if not os.path.exists('.env'):
        with open('.env.template', 'w') as f:
            f.write(env_template)
        print("Created .env.template file. Please add your API keys and rename to .env")
    
    # Load environment variables if .env exists
    if os.path.exists('.env'):
        load_dotenv()
        print("Environment variables loaded!")
    
    print("Setup complete!")

setup_environment()

# Cell 4: Create knowledge base from custom documents
class KnowledgeBaseBuilder:
    """Build and manage knowledge base for RAG"""
    
    def __init__(self, persist_directory="./chroma_db"):
        self.persist_directory = persist_directory
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )
        self.vectorstore = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    
    def create_sample_documents(self):
        """Create sample knowledge base documents"""
        documents = [
            Document(
                page_content="""
                Artificial Intelligence (AI) is the simulation of human intelligence in machines 
                that are programmed to think and learn. Machine learning is a subset of AI that 
                enables systems to learn and improve from experience without being explicitly programmed. 
                Deep learning is a subset of machine learning that uses neural networks with multiple layers.
                """,
                metadata={"source": "ai_basics", "topic": "artificial intelligence"}
            ),
            Document(
                page_content="""
                Machine Learning algorithms can be categorized into three main types: 
                supervised learning, unsupervised learning, and reinforcement learning. 
                Supervised learning uses labeled data, unsupervised learning finds patterns 
                in unlabeled data, and reinforcement learning learns through trial and error.
                """,
                metadata={"source": "ml_types", "topic": "machine learning"}
            ),
            Document(
                page_content="""
                Natural Language Processing (NLP) is a branch of AI that helps computers 
                understand, interpret, and manipulate human language. Key applications include 
                sentiment analysis, named entity recognition, machine translation, and chatbots.
                """,
                metadata={"source": "nlp_basics", "topic": "natural language processing"}
            ),
            Document(
                page_content="""
                LangChain is a framework for developing applications powered by language models. 
                It provides tools for building RAG systems, managing memory, creating chains, 
                and integrating with various LLMs. LangChain supports multiple model providers 
                including OpenAI, Groq, Anthropic, and local models via Ollama.
                """,
                metadata={"source": "langchain_intro", "topic": "langchain"}
            ),
            Document(
                page_content="""
                Retrieval-Augmented Generation (RAG) is a technique that combines information 
                retrieval with text generation. It enhances LLM responses by retrieving relevant 
                documents from a knowledge base and using them as context. RAG improves accuracy, 
                reduces hallucinations, and enables LLMs to access external knowledge.
                """,
                metadata={"source": "rag_concept", "topic": "retrieval augmented generation"}
            ),
            Document(
                page_content="""
                Vector databases are specialized databases designed to store and search 
                vector embeddings. Popular vector databases include Chroma, FAISS, Pinecone, 
                and Weaviate. They enable semantic similarity search by comparing vector 
                distances using metrics like cosine similarity or Euclidean distance.
                """,
                metadata={"source": "vector_dbs", "topic": "vector databases"}
            ),
            Document(
                page_content="""
                Python is a high-level, interpreted programming language known for its 
                simplicity and readability. It is widely used for data science, machine learning, 
                web development, and automation. Python's extensive ecosystem includes libraries 
                like NumPy, Pandas, Scikit-learn, and PyTorch.
                """,
                metadata={"source": "python_intro", "topic": "programming"}
            ),
            Document(
                page_content="""
                Cloud computing provides on-demand access to computing resources over the internet. 
                Major cloud providers include AWS (Amazon Web Services), Microsoft Azure, and 
                Google Cloud Platform (GCP). Cloud services include compute, storage, databases, 
                and AI/ML services.
                """,
                metadata={"source": "cloud_computing", "topic": "cloud"}
            )
        ]
        
        return documents
    
    def load_documents_from_files(self, file_paths: List[str]):
        """Load documents from files (PDF, TXT)"""
        documents = []
        
        for file_path in file_paths:
            if file_path.endswith('.pdf'):
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            elif file_path.endswith('.txt'):
                loader = TextLoader(file_path)
                documents.extend(loader.load())
            else:
                print(f"Unsupported file type: {file_path}")
        
        return documents
    
    def load_from_web(self, urls: List[str]):
        """Load documents from web URLs"""
        documents = []
        
        for url in urls:
            try:
                loader = WebBaseLoader(url)
                documents.extend(loader.load())
                print(f"Loaded: {url}")
            except Exception as e:
                print(f"Error loading {url}: {e}")
        
        return documents
    
    def build_vectorstore(self, documents: List[Document]):
        """Build vector store from documents"""
        # Split documents into chunks
        chunks = self.text_splitter.split_documents(documents)
        print(f"Created {len(chunks)} chunks from {len(documents)} documents")
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.persist_directory
        )
        
        # Persist the vector store
        self.vectorstore.persist()
        print(f"Vector store saved to {self.persist_directory}")
        
        return self.vectorstore
    
    def load_existing_vectorstore(self):
        """Load existing vector store"""
        self.vectorstore = Chroma(
            persist_directory=self.persist_directory,
            embedding_function=self.embeddings
        )
        print(f"Loaded vector store from {self.persist_directory}")
        return self.vectorstore
    
    def search_similar(self, query: str, k: int = 3):
        """Search for similar documents"""
        if self.vectorstore:
            results = self.vectorstore.similarity_search_with_score(query, k=k)
            return results
        return []

# Cell 5: Build the knowledge base
print("\n=== Building Knowledge Base ===")

# Initialize knowledge base builder
kb_builder = KnowledgeBaseBuilder(persist_directory="./knowledge_chroma_db")

# Create sample documents
sample_docs = kb_builder.create_sample_documents()
print(f"Created {len(sample_docs)} sample documents")

# Build vector store
vectorstore = kb_builder.build_vectorstore(sample_docs)

# Test search
test_query = "What is RAG?"
results = kb_builder.search_similar(test_query, k=2)

print(f"\nTest search for '{test_query}':")
for doc, score in results:
    print(f"  Score: {score:.4f}")
    print(f"  Content: {doc.page_content[:150]}...")
    print(f"  Source: {doc.metadata.get('source', 'unknown')}\n")

# Cell 6: Create the chatbot with memory
class ContextAwareChatbot:
    """Chatbot with conversation memory and RAG capabilities"""
    
    def __init__(self, vectorstore, model_name="llama-3.1-8b-instant", use_groq=True):
        self.vectorstore = vectorstore
        self.model_name = model_name
        self.use_groq = use_groq
        self.llm = None
        self.qa_chain = None
        self.conversation_memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="answer"
        )
        
        # Initialize LLM
        self._initialize_llm()
        
        # Create chain
        self._create_chain()
    
    def _initialize_llm(self):
        """Initialize the language model"""
        if self.use_groq:
            # Use Groq for faster inference (requires API key)
            api_key = os.getenv("GROQ_API_KEY")
            if api_key:
                self.llm = ChatGroq(
                    model=self.model_name,
                    api_key=api_key,
                    temperature=0.7,
                    streaming=False
                )
                print(f"Initialized Groq LLM with model: {self.model_name}")
            else:
                print("GROQ_API_KEY not found. Falling back to Ollama...")
                self._setup_ollama()
        else:
            self._setup_ollama()
    
    def _setup_ollama(self):
        """Setup Ollama for local LLM"""
        try:
            self.llm = ChatOllama(
                model="llama3.2",
                temperature=0.7,
                callbacks=[StreamingStdOutCallbackHandler()]
            )
            print("Initialized Ollama LLM with model: llama3.2")
        except Exception as e:
            print(f"Error initializing Ollama: {e}")
            print("Please ensure Ollama is installed and llama3.2 model is pulled")
    
    def _create_chain(self):
        """Create the conversational retrieval chain"""
        
        # Custom prompt template
        prompt_template = """You are a helpful AI assistant with access to a knowledge base. 
        Use the following pieces of context to answer the user's question. 
        If you don't know the answer based on the context, say so clearly.
        Keep answers concise and relevant to the question.
        
        Context from knowledge base:
        {context}
        
        Chat History:
        {chat_history}
        
        Current Question: {question}
        
        Helpful Answer:"""
        
        # Create prompt
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", prompt_template),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
        ])
        
        # Create retriever from vectorstore
        retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 4}
        )
        
        # Create the conversational chain
        self.qa_chain = ConversationalRetrievalChain.from_llm(
            llm=self.llm,
            retriever=retriever,
            memory=self.conversation_memory,
            return_source_documents=True,
            verbose=False
        )
        
        print("Conversational chain created successfully!")
    
    def chat(self, message: str) -> Dict[str, Any]:
        """Process a user message and return response"""
        try:
            response = self.qa_chain({"question": message})
            
            return {
                "answer": response["answer"],
                "source_documents": response.get("source_documents", []),
                "success": True
            }
        except Exception as e:
            return {
                "answer": f"Sorry, I encountered an error: {str(e)}",
                "source_documents": [],
                "success": False
            }
    
    def get_conversation_history(self):
        """Get the conversation history"""
        return self.conversation_memory.chat_memory.messages
    
    def clear_memory(self):
        """Clear conversation memory"""
        self.conversation_memory.clear()
        print("Conversation memory cleared!")

# Cell 7: Advanced memory management
class AdvancedMemoryChatbot(ContextAwareChatbot):
    """Chatbot with both short-term and long-term memory"""
    
    def __init__(self, vectorstore, model_name="llama-3.1-8b-instant", use_groq=True):
        super().__init__(vectorstore, model_name, use_groq)
        
        # Long-term memory using vector store
        self.long_term_memory = VectorStoreRetrieverMemory(
            retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
            memory_key="long_term_context",
            return_docs=True
        )
        
        # Short-term memory buffer with window
        self.short_term_memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            k=5  # Keep last 5 exchanges
        )
    
    def chat_with_long_term_memory(self, message: str) -> Dict[str, Any]:
        """Chat using both short-term and long-term memory"""
        
        # Retrieve relevant long-term memories
        long_term_context = self.long_term_memory.load_memory_variables({"input": message})
        
        # Get short-term history
        short_term_history = self.short_term_memory.load_memory_variables({})
        
        # Enhanced prompt with both memory types
        enhanced_prompt = f"""
        Long-term knowledge from past conversations:
        {long_term_context.get('long_term_context', 'No relevant long-term memory')}
        
        Recent conversation history:
        {short_term_history.get('chat_history', 'No recent conversation')}
        
        User question: {message}
        
        Please provide a helpful response considering both the long-term context and recent conversation.
        """
        
        # Store in short-term memory
        self.short_term_memory.save_context({"input": message}, {"output": "processing..."})
        
        # Get response using base method
        response = self.chat(enhanced_prompt)
        
        # Update short-term memory with actual response
        if response["success"]:
            messages = self.short_term_memory.chat_memory.messages
            if len(messages) >= 2:
                messages[-1].content = response["answer"]
        
        return response

# Cell 8: Initialize the chatbot
print("\n=== Initializing Chatbot ===")

# Check if vector store exists
if os.path.exists("./knowledge_chroma_db"):
    print("Loading existing vector store...")
    kb_builder = KnowledgeBaseBuilder("./knowledge_chroma_db")
    vectorstore = kb_builder.load_existing_vectorstore()
else:
    print("Building new vector store...")
    kb_builder = KnowledgeBaseBuilder("./knowledge_chroma_db")
    sample_docs = kb_builder.create_sample_documents()
    vectorstore = kb_builder.build_vectorstore(sample_docs)

# Initialize chatbot
print("\nInitializing chatbot with Groq (or fallback to Ollama)...")
chatbot = ContextAwareChatbot(
    vectorstore=vectorstore,
    model_name="llama-3.1-8b-instant",
    use_groq=True
)

print("\nChatbot ready! You can now start a conversation.")

# Cell 9: Interactive chat function for notebook
def interactive_chat():
    """Run interactive chat in notebook"""
    print("\n" + "="*60)
    print("Context-Aware Chatbot - Interactive Mode")
    print("="*60)
    print("Type 'quit' to exit, 'clear' to clear memory, 'history' to see conversation")
    print("-"*60)
    
    while True:
        user_input = input("\nYou: ").strip()
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        elif user_input.lower() == 'clear':
            chatbot.clear_memory()
            print("Conversation memory cleared!")
            continue
        elif user_input.lower() == 'history':
            history = chatbot.get_conversation_history()
            print("\nConversation History:")
            for msg in history:
                print(f"{msg.type}: {msg.content[:100]}...")
            continue
        
        if user_input:
            print("Assistant: ", end="", flush=True)
            response = chatbot.chat(user_input)
            print(response["answer"])
            
            # Show sources if available
            if response["source_documents"]:
                print("\n[Sources:]")
                for i, doc in enumerate(response["source_documents"][:2], 1):
                    source = doc.metadata.get('source', 'unknown')
                    print(f"  {i}. {source}")

# Uncomment to run interactive chat
# interactive_chat()

# Cell 10: Test the chatbot with sample questions
print("\n=== Testing Chatbot ===")

test_questions = [
    "What is RAG and how does it work?",
    "Tell me about LangChain framework",
    "What are vector databases used for?",
    "Can you explain the difference between AI and Machine Learning?",
    "What was my first question?"  # Tests memory
]

print("\nRunning test queries:")
for question in test_questions:
    print(f"\nQ: {question}")
    response = chatbot.chat(question)
    print(f"A: {response['answer'][:200]}...")
    if response["source_documents"]:
        print(f"  (Retrieved {len(response['source_documents'])} sources)")

# Cell 11: Streamlit app code (save as app.py)
streamlit_app_code = """
# ============================================
# Streamlit App for Context-Aware Chatbot
# Save this as 'chatbot_app.py'
# ============================================

import streamlit as st
import sys
import os

# Add parent directory to path if needed
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

# Import the chatbot class
from your_notebook import ContextAwareChatbot, KnowledgeBaseBuilder

def initialize_session_state():
    """Initialize session state variables"""
    if "chatbot" not in st.session_state:
        # Load or create vector store
        kb_builder = KnowledgeBaseBuilder("./knowledge_chroma_db")
        if os.path.exists("./knowledge_chroma_db"):
            vectorstore = kb_builder.load_existing_vectorstore()
        else:
            documents = kb_builder.create_sample_documents()
            vectorstore = kb_builder.build_vectorstore(documents)
        
        # Initialize chatbot
        st.session_state.chatbot = ContextAwareChatbot(
            vectorstore=vectorstore,
            model_name="llama-3.1-8b-instant",
            use_groq=True
        )
        st.session_state.messages = []
        st.session_state.conversation_active = True

def main():
    st.set_page_config(
        page_title="Context-Aware Chatbot",
        page_icon="🤖",
        layout="wide"
    )
    
    st.title("🤖 Context-Aware Chatbot")
    st.markdown("A chatbot with memory and knowledge retrieval capabilities")
    
    # Sidebar
    with st.sidebar:
        st.header("Settings")
        
        # Clear conversation button
        if st.button("Clear Conversation"):
            st.session_state.chatbot.clear_memory()
            st.session_state.messages = []
            st.rerun()
        
        # Show conversation history toggle
        show_history = st.checkbox("Show conversation history", value=False)
        
        # Model info
        st.header("About")
        st.info("""
        This chatbot features:
        - Conversation memory
        - Knowledge base retrieval (RAG)
        - Contextual responses
        - Source attribution
        """)
        
        # Add custom knowledge button
        st.header("Add Knowledge")
        uploaded_file = st.file_uploader("Upload document", type=["txt", "pdf"])
        if uploaded_file is not None:
            st.success(f"File uploaded: {uploaded_file.name}")
            st.info("Knowledge base update coming soon!")
    
    # Display conversation history
    if show_history and st.session_state.messages:
        st.header("Conversation History")
        for msg in st.session_state.messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            with st.expander(f"{role}: {msg['content'][:100]}..."):
                st.write(msg["content"])
                if "sources" in msg and msg["sources"]:
                    st.write("**Sources:**")
                    for source in msg["sources"]:
                        st.write(f"- {source}")
    
    # Main chat interface
    st.header("Chat Interface")
    
    # Display chat messages
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.write(message["content"])
            if "sources" in message and message["sources"]:
                with st.expander("View sources"):
                    for source in message["sources"]:
                        st.write(f"- {source}")
    
    # Chat input
    if prompt := st.chat_input("Ask me anything about AI, ML, LangChain, RAG, or programming..."):
        # Add user message
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.write(prompt)
        
        # Get response
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                response = st.session_state.chatbot.chat(prompt)
                
                if response["success"]:
                    st.write(response["answer"])
                    
                    # Show sources
                    sources = []
                    if response["source_documents"]:
                        with st.expander("Sources"):
                            for i, doc in enumerate(response["source_documents"]):
                                source = doc.metadata.get('source', f'Document {i+1}')
                                st.write(f"- {source}")
                                sources.append(source)
                    
                    # Add assistant message
                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": response["answer"],
                        "sources": sources
                    })
                else:
                    st.error(response["answer"])
                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": response["answer"]
                    })

if __name__ == "__main__":
    main()
"""

# Save Streamlit app
with open('chatbot_app.py', 'w') as f:
    f.write(streamlit_app_code)
print("\nStreamlit app saved as 'chatbot_app.py'")
print("Run with: streamlit run chatbot_app.py")

# Cell 12: Test different memory configurations
print("\n=== Memory Configuration Test ===")

class MemoryTest:
    """Test different memory configurations"""
    
    @staticmethod
    def test_buffer_memory():
        """Test ConversationBufferMemory"""
        from langchain.memory import ConversationBufferMemory
        
        memory = ConversationBufferMemory(return_messages=True)
        memory.save_context({"input": "Hello, my name is Alice"}, {"output": "Hi Alice! Nice to meet you."})
        memory.save_context({"input": "I work as a data scientist"}, {"output": "That's interesting!"})
        
        print("\nBuffer Memory (stores all):")
        print(f"Number of messages: {len(memory.chat_memory.messages)}")
        print(f"Messages: {[msg.content for msg in memory.chat_memory.messages]}")
        
        return memory
    
    @staticmethod
    def test_window_memory():
        """Test ConversationBufferWindowMemory"""
        from langchain.memory import ConversationBufferWindowMemory
        
        memory = ConversationBufferWindowMemory(k=2, return_messages=True)
        
        # Add multiple messages
        for i in range(5):
            memory.save_context(
                {"input": f"Message {i}"},
                {"output": f"Response to message {i}"}
            )
        
        print("\nWindow Memory (keeps last 2):")
        print(f"Number of messages: {len(memory.chat_memory.messages)}")
        print(f"Messages: {[msg.content for msg in memory.chat_memory.messages]}")
        
        return memory

MemoryTest.test_buffer_memory()
MemoryTest.test_window_memory()

# Cell 13: RAG retrieval evaluation
print("\n=== RAG Retrieval Evaluation ===")

def evaluate_retrieval(chatbot, test_queries):
    """Evaluate retrieval quality"""
    results = []
    
    for query in test_queries:
        # Search for relevant documents
        results_search = chatbot.qa_chain.retriever.get_relevant_documents(query)
        
        # Get response
        response = chatbot.chat(query)
        
        results.append({
            "query": query,
            "retrieved_chunks": len(results_search),
            "has_answer": response["success"],
            "answer_length": len(response["answer"])
        })
    
    # Create summary DataFrame
    df = pd.DataFrame(results)
    print("\nRetrieval Evaluation Summary:")
    print(df.to_string(index=False))
    print(f"\nAverage retrieved chunks: {df['retrieved_chunks'].mean():.1f}")
    print(f"Success rate: {df['has_answer'].sum()}/{len(df)} ({df['has_answer'].sum()/len(df)*100:.0f}%)")
    
    return df

# Test retrieval
test_queries = [
    "What is LangChain?",
    "Explain RAG",
    "How do vector databases work?",
    "What is machine learning?"
]

eval_results = evaluate_retrieval(chatbot, test_queries)

# Cell 14: Custom prompt template examples
print("\n=== Custom Prompt Templates ===")

class PromptTemplates:
    """Collection of custom prompt templates for different use cases"""
    
    @staticmethod
    def get_general_prompt():
        """General purpose prompt template"""
        return """You are a helpful AI assistant with access to a knowledge base.
        
        Context: {context}
        
        Chat History: {chat_history}
        
        Question: {question}
        
        Provide a clear, concise answer based on the context.
        If unsure, say "I don't have enough information to answer that."
        """
    
    @staticmethod
    def get_technical_prompt():
        """Technical/educational prompt template"""
        return """You are a technical expert explaining complex concepts.
        
        Relevant information: {context}
        
        Previous discussion: {chat_history}
        
        User question: {question}
        
        Guidelines:
        1. Use examples when helpful
        2. Break down complex ideas
        3. Include relevant technical terms
        4. Suggest follow-up topics
        
        Answer:
        """
    
    @staticmethod
    def get_support_prompt():
        """Customer support style prompt"""
        return """You are a knowledgeable support agent.
        
        Knowledge base articles: {context}
        
        Conversation so far: {chat_history}
        
        Customer inquiry: {question}
        
        Provide:
        1. Direct answer to the question
        2. Any relevant additional information
        3. Clear next steps if applicable
        
        Response:
        """
    
    @staticmethod
    def compare_prompts(chatbot, question):
        """Compare different prompt templates"""
        templates = {
            "General": PromptTemplates.get_general_prompt(),
            "Technical": PromptTemplates.get_technical_prompt(),
            "Support": PromptTemplates.get_support_prompt()
        }
        
        print(f"\nComparing prompts for: '{question}'")
        print("-"*50)
        
        for name, template in templates.items():
            print(f"\n{name} Prompt Response:")
            print(f"Template: {template[:100]}...")
        
        return templates

# Test prompt comparison
test_question = "What are the benefits of RAG?"
PromptTemplates.compare_prompts(chatbot, test_question)

# Cell 15: Export chatbot for production
import pickle
import json

def export_chatbot(chatbot, export_path="./exported_chatbot"):
    """Export chatbot configuration and artifacts"""
    
    os.makedirs(export_path, exist_ok=True)
    
    # Export configuration
    config = {
        "model_name": chatbot.model_name,
        "use_groq": chatbot.use_groq,
        "vectorstore_path": "./knowledge_chroma_db",
        "memory_type": "ConversationBufferMemory"
    }
    
    with open(f"{export_path}/config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    # Export sample conversation starters
    starters = [
        "What is RAG?",
        "Explain LangChain memory",
        "How do I use vector databases?",
        "What are the benefits of conversational AI?",
        "Tell me about machine learning types"
    ]
    
    with open(f"{export_path}/conversation_starters.json", "w") as f:
        json.dump(starters, f, indent=2)
    
    print(f"Chatbot exported to {export_path}")
    print(f"Files created: config.json, conversation_starters.json")
    
    return export_path

# Export the chatbot
export_path = export_chatbot(chatbot)
print(f"\nChatbot exported successfully to: {export_path}")

# Cell 16: Deployment helper functions
def create_requirements_file():
    """Create requirements.txt for deployment"""
    requirements = """
streamlit>=1.28.0
langchain>=0.1.0
langchain-community>=0.0.10
langchain-groq>=0.1.0
chromadb>=0.4.0
sentence-transformers>=2.2.0
pypdf>=3.0.0
tiktoken>=0.5.0
python-dotenv>=1.0.0
pandas>=1.5.0
numpy>=1.24.0
    """
    
    with open("requirements.txt", "w") as f:
        f.write(requirements.strip())
    print("requirements.txt created!")

def create_dockerfile():
    """Create Dockerfile for container deployment"""
    dockerfile = """
FROM python:3.10-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY . .

# Expose Streamlit port
EXPOSE 8501

# Run Streamlit app
CMD ["streamlit", "run", "chatbot_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
    """
    
    with open("Dockerfile", "w") as f:
        f.write(dockerfile.strip())
    print("Dockerfile created!")

def create_deployment_script():
    """Create deployment script"""
    script = """#!/bin/bash
# Deployment script for Context-Aware Chatbot

echo "Deploying Context-Aware Chatbot..."

# Check Python version
python_version=$(python3 --version)
echo "Python version: $python_version"

# Install dependencies
echo "Installing dependencies..."
pip install -r requirements.txt

# Create .env file from template if not exists
if [ ! -f .env ]; then
    cp .env.template .env
    echo "Created .env file. Please add your API keys."
fi

# Run the Streamlit app
echo "Starting Streamlit app..."
streamlit run chatbot_app.py --server.port=8501 --server.address=0.0.0.0
"""
    
    with open("deploy.sh", "w") as f:
        f.write(script)
    print("deploy.sh created!")

# Create deployment files
create_requirements_file()
create_dockerfile()
create_deployment_script()

print("\nDeployment files created successfully!")

# Cell 17: Final summary and testing
print("\n" + "="*60)
print("PROJECT COMPLETED SUCCESSFULLY")
print("="*60)

print("\nCONTEXT-AWARE CHATBOT SUMMARY")
print("-"*60)
print("\nFeatures Implemented:")
print("  1. Knowledge Base: 8 sample documents about AI/ML/LangChain/RAG")
print("  2. Vector Store: Chroma with sentence-transformers embeddings")
print("  3. Memory: ConversationBufferMemory for short-term context")
print("  4. RAG: Retrieval-augmented generation with source attribution")
print("  5. LLM Support: Groq (primary) and Ollama (fallback)")
print("  6. Web Interface: Streamlit app with chat interface")
print("  7. Deployment: Docker + requirements.txt + deployment script")

print("\nFile Structure Created:")
print("  - knowledge_chroma_db/     (Vector store with embeddings)")
print("  - chatbot_app.py           (Streamlit web application)")
print("  - requirements.txt         (Python dependencies)")
print("  - Dockerfile              (Container configuration)")
print("  - deploy.sh               (Deployment script)")
print("  - exported_chatbot/       (Exported configuration)")

print("\nHow to Run:")
print("  1. Web App: streamlit run chatbot_app.py")
print("  2. Interactive: Run the interactive_chat() function")
print("  3. Docker: docker build -t chatbot . && docker run -p 8501:8501 chatbot")
print("  4. Cloud: Deploy to Streamlit Cloud, Hugging Face Spaces, or AWS")

print("\nExample Conversations to Try:")
examples = [
    "What is RAG and why is it important?",
    "Explain how LangChain manages conversation memory",
    "What are vector databases and how do they work?",
    "Can you summarize what we've discussed so far?"
]
for i, example in enumerate(examples, 1):
    print(f"  {i}. {example}")

print("\n" + "="*60)
print("The chatbot is ready for deployment!")
print("="*60)